# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then inspect token embeddings before adding attention.

In [1]:
import random
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [2]:
tokens, vocab, merges = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
head_size = 16

random.seed(42)
x_batch, y_batch = get_batch(
    "train", train_tokens, validation_tokens, block_size, batch_size
)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Model scaffold

In [3]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        token_embeddings = self.token_embedding_table(idx)
        return self.lm_head(token_embeddings)

    # # x: [B, T, C]
    # q,k,v: [B, T, H]
    # out: [B, T, H]
    class Head(nn.Module):
        def __init__(self, n_embd, head_size):
            super().__init__()
            self.n_embd = n_embd
            self.head_size = head_size
            self.query = nn.Linear(n_embd, head_size)
            self.key = nn.Linear(n_embd, head_size)
            self.value = nn.Linear(n_embd, head_size)


model = AttentionLanguageModel(vocab_size, n_embd)
token_embeddings = model.token_embedding_table(x_batch)
print(f"Token embedding shape (B, T, C): {tuple(token_embeddings.shape)}")

Token embedding shape (B, T, C): (32, 8, 32)


## TODO — first attention mechanics

1. Add positional embeddings so each token position has its own learned representation.
2. Implement one causal self-attention head over the `(B, T, C)` token embeddings.

Stop here before adding either piece.

In [4]:
# Positional embeddings
positional_embeddings = nn.Parameter(torch.zeros(1, block_size, n_embd))
# Add positional embeddings to token embeddings
token_embeddings += positional_embeddings
print(f"Token embedding shape after adding positional embeddings (B, T, C): {tuple(token_embeddings.shape)}")

Token embedding shape after adding positional embeddings (B, T, C): (32, 8, 32)


In [5]:
# Causal self-attention
import math
import torch
from torch import nn

key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)
value = nn.Linear(n_embd, head_size, bias=False)

# x       [B, T, C]
# q,k,v   [B, T, H]
# scores  [B, T, T]
# weights [B, T, T]
# out     [B, T, H]
# MATCH THE SHAPES OF THE LINEAR LAYERS!! 
def causal_self_attention(x, mask=None):
    k = key(x)
    q = query(x)
    v = value(x)

    scores = q @ k.transpose(-1, -2) / math.sqrt(head_size)
    scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    attention_out = weights @ v
    return attention_out

x = torch.randn(1, 8, n_embd)
mask = torch.ones(1, 8, 8)
causal_self_attention(x, mask)

# multiple heads


tensor([[[-0.3359, -0.0926, -0.5004,  0.3693,  0.1200,  0.1665, -0.2064,
           0.0707, -0.4374, -0.2244, -0.0215,  0.1065, -0.3816, -0.0106,
          -0.0517,  0.2973],
         [-0.2948,  0.1323, -0.3300,  0.3204,  0.0064, -0.0053, -0.0520,
          -0.1661,  0.0267, -0.2100, -0.1776,  0.3888, -0.2516,  0.2775,
           0.1688,  0.3433],
         [-0.4002,  0.1070, -0.2380,  0.2183, -0.0651, -0.0246, -0.0918,
          -0.0476, -0.1315, -0.3247, -0.1048,  0.3659, -0.2446,  0.3012,
          -0.0368,  0.2131],
         [-0.2986,  0.0866, -0.3478,  0.2927,  0.0481,  0.0320, -0.1348,
          -0.0347, -0.2333, -0.2642, -0.1208,  0.2165, -0.2372,  0.1028,
          -0.0316,  0.2399],
         [-0.3288,  0.0568, -0.3325,  0.4484,  0.0970, -0.0041, -0.0488,
          -0.0714, -0.2942, -0.4379,  0.0574,  0.2957, -0.3744,  0.1609,
           0.0088,  0.2127],
         [-0.3845, -0.0171, -0.3743,  0.3885,  0.1183,  0.0627, -0.1314,
           0.0655, -0.4921, -0.4429,  0.0943,  0.165